# Portfolio of Evidence: Convolutional Neural Networks (CNN)
**Scenario:** The publisher wants to use image recognition (facial recognition later) at all premises. As a first step, this notebook trains a CNN that accurately classifies images.

**Student name / number:** ______________  **Module:** ______________

## 1. Dataset choice: evaluation and justification

**Chosen dataset: CIFAR-10** (60 000 colour images, 32x32 px, 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck).

| Criterion | Assessment |
|---|---|
| **Quality** | Professionally labelled, widely benchmarked and peer-cited. No missing values. Perfectly balanced (6 000 images per class). Fixed official train (50 000) / test (10 000) split prevents leakage. |
| **Suitability** | Colour images with varied backgrounds, lighting and poses, which is closer to real premises footage than centred digits (MNIST). Small enough to train on a laptop or free Colab GPU. Harder than MNIST, so it demonstrates real CNN value. |
| **Size** | 50 000 training images is enough for a small CNN without pre-training. |
| **Limitations (be honest in your report)** | Low resolution (32x32). Classes are objects, not faces. Faces would need a dataset such as LFW or CelebA and privacy/POPIA consent. Some images may be near-duplicates (checked below). |
| **Alternatives considered** | MNIST (too easy, greyscale), Fashion-MNIST (greyscale), CelebA/LFW (faces, but large, privacy-sensitive, and the brief says faces are not required yet). |

> If your module supplied a specific dataset, swap it in at the loading cell and keep the same justification structure.

## 2. Analysis plan (succinct)

| Stage | Plan |
|---|---|
| **EDA** | Check shape/dtype, missing or corrupt values, pixel range, duplicates and train/test leakage. Plot class balance, sample images, mean image per class, RGB intensity distributions, brightness per class. |
| **Feature selection** | CNNs learn features automatically, so P-value forward/backward elimination (a regression technique) does not apply to raw pixels. Plan: scale pixels to 0-1, check per-channel variance, and use data augmentation. Then justify keeping all 3 channels. |
| **Train model** | Baseline: 3 conv blocks (32/64/128 filters), 3x3 kernels, ReLU, max-pool, dense 128, softmax. Adam (lr 0.001), batch 64, 15 epochs, sparse categorical cross-entropy, 10% validation split. |
| **Evaluate** | Accuracy (balanced classes, so it is valid), loss curves (over/underfitting), per-class precision/recall/F1, confusion matrix. |
| **Improve** | If overfitting: add BatchNorm, Dropout, data augmentation, early stopping, LR reduction. Compare against the baseline. |
| **Report** | Data overview, cleaning and EDA findings, architecture and hyperparameters, metrics, baseline vs improved comparison, limitations and recommendations. |

## 3. Setup

In [ ]:
# Import libraries: numpy/pandas for data handling, matplotlib/seaborn for visuals,
# tensorflow.keras for building the CNN, sklearn for evaluation metrics.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

# Set seeds so results are reproducible
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
sns.set_style("whitegrid")

In [ ]:
# Load CIFAR-10 (downloads automatically the first time).
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Labels come as shape (n,1); flatten to (n,) for easier handling
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print("Train images:", x_train_full.shape, "| Train labels:", y_train_full.shape)
print("Test images :", x_test.shape, "| Test labels :", y_test.shape)
print("Data type:", x_train_full.dtype)

## 4. Exploratory Data Analysis (EDA) and data cleaning

In [ ]:
# 4.1 Data quality: missing values, pixel range, label validity
print("Missing values (train):", np.isnan(x_train_full.astype(float)).sum())
print("Pixel min/max:", x_train_full.min(), x_train_full.max(), "(expected 0-255)")
print("Unique labels:", np.unique(y_train_full), "(expected 0-9)")
print("Image shape consistent:", x_train_full.shape[1:] == x_test.shape[1:])

In [ ]:
# 4.2 Duplicate and leakage check using a hash of each image's raw bytes.
# Duplicates inflate accuracy if the same image is in both train and test.
def img_hash(img): return hashlib.md5(img.tobytes()).hexdigest()

train_hashes = [img_hash(i) for i in x_train_full]
test_hashes  = [img_hash(i) for i in x_test]

dup_within_train = len(train_hashes) - len(set(train_hashes))
leakage = len(set(train_hashes) & set(test_hashes))
print("Exact duplicates inside train:", dup_within_train)
print("Images appearing in BOTH train and test (leakage):", leakage)

# Cleaning step: drop exact duplicates in training data (keep first occurrence)
_, unique_idx = np.unique(train_hashes, return_index=True)
unique_idx = np.sort(unique_idx)
x_train_full = x_train_full[unique_idx]
y_train_full = y_train_full[unique_idx]
print("Training images after de-duplication:", len(x_train_full))

In [ ]:
# 4.3 Class balance: an imbalanced dataset would make accuracy misleading.
counts = pd.Series(y_train_full).map(dict(enumerate(class_names))).value_counts().sort_index()
plt.figure(figsize=(9,4))
sns.barplot(x=counts.index, y=counts.values, color="steelblue")
plt.title("Training images per class"); plt.xticks(rotation=45); plt.ylabel("Count")
plt.tight_layout(); plt.show()
print(counts)

In [ ]:
# 4.4 Sample images: visual sanity check that labels match content
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for k, ax in enumerate(axes.flat):
    idx = np.where(y_train_full == k)[0][0]
    ax.imshow(x_train_full[idx]); ax.set_title(class_names[k]); ax.axis("off")
plt.suptitle("One example per class"); plt.tight_layout(); plt.show()

In [ ]:
# 4.5 Mean image per class: shows the average "pattern" the CNN can learn
# (e.g. ships/airplanes have blue backgrounds, frogs/deer green-brown).
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for k, ax in enumerate(axes.flat):
    mean_img = x_train_full[y_train_full == k].mean(axis=0) / 255.0
    ax.imshow(mean_img); ax.set_title(class_names[k]); ax.axis("off")
plt.suptitle("Mean image per class"); plt.tight_layout(); plt.show()

In [ ]:
# 4.6 Pixel intensity distribution per colour channel
plt.figure(figsize=(9,4))
for ch, col in enumerate(["red","green","blue"]):
    plt.hist(x_train_full[:, :, :, ch].ravel(), bins=50, alpha=0.5, color=col, label=col)
plt.title("Pixel intensity distribution by channel"); plt.xlabel("Pixel value (0-255)")
plt.legend(); plt.show()

# Per-channel mean and standard deviation (used to justify normalisation)
print("Channel means:", (x_train_full.mean(axis=(0,1,2))).round(1))
print("Channel std  :", (x_train_full.std(axis=(0,1,2))).round(1))

In [ ]:
# 4.7 Brightness per class: does lighting differ between classes? If so the model
# could "cheat" by using brightness instead of the object itself.
brightness = x_train_full.mean(axis=(1,2,3))
df_b = pd.DataFrame({"brightness": brightness, "class": [class_names[i] for i in y_train_full]})
plt.figure(figsize=(11,4))
sns.boxplot(data=df_b, x="class", y="brightness", color="lightsteelblue")
plt.xticks(rotation=45); plt.title("Image brightness by class"); plt.tight_layout(); plt.show()

### EDA findings (edit with your actual output)
- No missing or corrupt values. All pixels in the 0-255 range, and all images are 32x32x3.
- Classes are balanced, so **accuracy is a fair headline metric**.
- Any exact duplicates and train/test leakage are reported by the check above.
- Colour patterns differ by class (e.g. ships/airplanes lean blue). The CNN can use this, but brightness overlaps heavily between classes, so it is not a shortcut.
- Pixel values need scaling, as raw 0-255 values slow and destabilise training.

## 5. Feature selection and preparation

In [ ]:
# CNNs perform automatic feature extraction: the convolution filters learn edges,
# textures and shapes themselves. P-value elimination is for tabular regression, not raw pixels.
# Our "feature selection" is therefore:
#   (1) check whether any colour channel is uninformative
#   (2) scale/normalise the input
#   (3) augment data to improve the usefulness of the features learned

# (1) Channel check: variance per channel and accuracy of a quick model per channel
print("Variance per channel:", x_train_full.var(axis=(0,1,2)).round(1))

In [ ]:
# Quick experiment: a tiny CNN on each channel alone vs all 3 channels.
# If RGB clearly beats any single channel, we keep all three.
def quick_cnn(channels_idx, epochs=3):
    xtr = x_train_full[:, :, :, channels_idx].astype("float32")/255.0
    xte = x_test[:, :, :, channels_idx].astype("float32")/255.0
    m = keras.Sequential([
        layers.Input(shape=xtr.shape[1:]),
        layers.Conv2D(16, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Flatten(), layers.Dense(10, activation="softmax")])
    m.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(xtr, y_train_full, epochs=epochs, batch_size=128, verbose=0)
    return m.evaluate(xte, y_test, verbose=0)[1]

res = {"Red": quick_cnn([0]), "Green": quick_cnn([1]), "Blue": quick_cnn([2]),
       "RGB (all)": quick_cnn([0,1,2])}
print(pd.Series(res).round(3))
# Conclusion: keep all 3 channels if RGB scores highest.

In [ ]:
# (2) Normalise to 0-1 and split off a validation set (10%) from the training data.
# The test set stays untouched until final evaluation.
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.1, stratify=y_train_full, random_state=SEED)
print("Train:", x_train.shape, "Validation:", x_val.shape, "Test:", x_test.shape)

## 6. Train the baseline model

In [ ]:
# Baseline CNN: 3 convolution blocks extract increasingly complex features, then
# a dense classifier head maps them to 10 class probabilities.
def build_baseline():
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, (3,3), activation="relu", padding="same"),  # edges/colours
        layers.MaxPooling2D((2,2)),                                    # shrink 32->16
        layers.Conv2D(64, (3,3), activation="relu", padding="same"),  # textures
        layers.MaxPooling2D((2,2)),                                    # 16->8
        layers.Conv2D(128, (3,3), activation="relu", padding="same"), # shapes/parts
        layers.MaxPooling2D((2,2)),                                    # 8->4
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax")                         # class probabilities
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy",  # integer labels
                  metrics=["accuracy"])
    return model

baseline = build_baseline()
baseline.summary()

In [ ]:
history_base = baseline.fit(x_train, y_train, epochs=15, batch_size=64,
                            validation_data=(x_val, y_val), verbose=2)

## 7. Evaluate the baseline

In [ ]:
# Learning curves: a growing gap between training and validation = overfitting.
def plot_history(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(12,4))
    ax[0].plot(h.history["accuracy"], label="train"); ax[0].plot(h.history["val_accuracy"], label="validation")
    ax[0].set_title(title + " - accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h.history["loss"], label="train"); ax[1].plot(h.history["val_loss"], label="validation")
    ax[1].set_title(title + " - loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()

plot_history(history_base, "Baseline")

In [ ]:
# Full evaluation on the held-out test set
def evaluate(model, name):
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
    print(f"{name}: test accuracy = {acc:.4f}, test loss = {loss:.4f}\n")
    print(classification_report(y_test, y_pred, target_names=class_names, digits=3))
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(9,7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion matrix - {name}")
    plt.tight_layout(); plt.show()
    return acc, y_pred

acc_base, pred_base = evaluate(baseline, "Baseline")

### Metric justification
- **Accuracy**: appropriate as a headline figure because classes are perfectly balanced.
- **Precision, recall, F1 per class**: show which classes fail (e.g. cat vs dog) and the cost of false positives vs false negatives.
- **Confusion matrix**: shows exactly which classes get confused, guiding improvements.
- **Loss and learning curves**: reveal overfitting that accuracy alone hides.

### Interpretation (edit with your actual numbers)
The baseline typically reaches about 70% test accuracy. Training accuracy keeps rising while validation accuracy plateaus and validation loss starts increasing after roughly epoch 5-8, which is **overfitting**. The weakest classes are normally cat, dog and bird, which look similar at 32x32. The strongest are automobile, ship and truck.

## 8. Retrain with improvements

**Changes made to address the baseline's shortcomings (overfitting)**

| Change | Reason |
|---|---|
| Double conv layers per block (VGG style) | Deeper feature extraction for subtle classes |
| Batch Normalisation | Stabilises and speeds up training |
| Dropout (0.25 conv, 0.5 dense) | Reduces overfitting |
| Data augmentation (flip, shift, rotate, zoom) | More varied training data without collecting more |
| EarlyStopping + ReduceLROnPlateau | Stops at the best epoch and fine-tunes the learning rate |
| Epochs 15 to 40 | More capacity to learn, controlled by early stopping |

In [ ]:
# Data augmentation is only active during training
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

def build_improved():
    model = keras.Sequential([
        layers.Input(shape=(32,32,3)),
        augment,
        # Block 1
        layers.Conv2D(32, 3, padding="same", activation="relu"), layers.BatchNormalization(),
        layers.Conv2D(32, 3, padding="same", activation="relu"), layers.BatchNormalization(),
        layers.MaxPooling2D(), layers.Dropout(0.25),
        # Block 2
        layers.Conv2D(64, 3, padding="same", activation="relu"), layers.BatchNormalization(),
        layers.Conv2D(64, 3, padding="same", activation="relu"), layers.BatchNormalization(),
        layers.MaxPooling2D(), layers.Dropout(0.25),
        # Block 3
        layers.Conv2D(128, 3, padding="same", activation="relu"), layers.BatchNormalization(),
        layers.Conv2D(128, 3, padding="same", activation="relu"), layers.BatchNormalization(),
        layers.MaxPooling2D(), layers.Dropout(0.25),
        # Classifier head
        layers.Flatten(),
        layers.Dense(128, activation="relu"), layers.BatchNormalization(), layers.Dropout(0.5),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(optimizer=keras.optimizers.Adam(0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

improved = build_improved()
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
]
history_imp = improved.fit(x_train, y_train, epochs=40, batch_size=64,
                           validation_data=(x_val, y_val), callbacks=callbacks, verbose=2)

In [ ]:
plot_history(history_imp, "Improved")
acc_imp, pred_imp = evaluate(improved, "Improved")

## 9. Compare models and inspect errors

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Baseline", "Improved"],
    "Test accuracy": [acc_base, acc_imp],
    "Final train acc": [history_base.history["accuracy"][-1], history_imp.history["accuracy"][-1]],
    "Final val acc": [history_base.history["val_accuracy"][-1], history_imp.history["val_accuracy"][-1]],
}).round(4)
comparison["Train-val gap"] = (comparison["Final train acc"] - comparison["Final val acc"]).round(4)
print(comparison.to_string(index=False))

In [ ]:
# Look at real mistakes: helps explain *why* the model fails
wrong = np.where(pred_imp != y_test)[0][:10]
fig, axes = plt.subplots(2, 5, figsize=(13,6))
for ax, i in zip(axes.flat, wrong):
    ax.imshow(x_test[i]); ax.axis("off")
    ax.set_title(f"True: {class_names[y_test[i]]}\nPred: {class_names[pred_imp[i]]}", fontsize=9)
plt.suptitle("Misclassified test images"); plt.tight_layout(); plt.show()

In [ ]:
# Save the final model for deployment/handover
improved.save("cifar10_cnn_improved.keras")

## 10. Report to the client

### 10.1 Purpose
The publisher wants image recognition across premises. This pilot proves a CNN can classify images accurately, as a stepping stone to facial recognition.

### 10.2 Data
CIFAR-10: 60 000 labelled 32x32 colour images in 10 balanced classes. *(Insert dataset justification from Section 1.)*

### 10.3 Data cleaning and EDA
*(Insert results.)* No missing or corrupt values, balanced classes, [n] duplicates removed, [n] train/test leaks. Charts: class balance, sample images, mean images, RGB histograms, brightness boxplot. Key insight: colour and background patterns differ by class, but brightness overlap is large, so the model cannot rely on lighting alone.

### 10.4 Model training
Baseline CNN of 3 conv blocks, Adam lr 0.001, batch 64, 15 epochs. Improved CNN with BatchNorm, Dropout, augmentation, EarlyStopping and LR scheduling. Inputs were scaled to 0-1, with a 90/10 train/validation split and a separate untouched test set.

### 10.5 Results and evaluation
*(Insert comparison table, curves, confusion matrix.)* Accuracy was chosen as the headline metric due to class balance, supported by per-class precision/recall/F1. The improved model reduced overfitting (smaller train-validation gap) and raised test accuracy from [x]% to [y]%.

### 10.6 Limitations and recommendations
- 32x32 images limit detail. Use higher-resolution data for production.
- Facial recognition needs a face dataset (e.g. LFW or CelebA), consent and POPIA compliance, and bias testing across demographic groups.
- Consider transfer learning (e.g. MobileNetV2, ResNet) for higher accuracy.
- Monitor performance after deployment and retrain as conditions change.